In [2]:

# =============================================================================
# GHOST SIGNAL AI v3.0
# Temporal Absence Intelligence & Event Investigation Platform
# COMPLETE SINGLE-CELL IMPLEMENTATION
#
# Modules:
#   01. Configuration management
#   02. Structured logging and audit trails
#   03. Synthetic event generation
#   04. Data validation and quality checks
#   05. Temporal sequence modeling
#   06. Missing-event and broken-transition detection
#   07. Isolation Forest anomaly detection
#   08. Explainable evidence-based investigation
#   09. Experiment tracking and evaluation
#   10. Automated testing
#   11. Model persistence and version metadata
#   12. Interactive visual analytics
#   13. CSV/JSON/ZIP artifact exports
#
# All data is synthetic. This is a runnable research prototype, not a
# validated production incident-detection system.
# =============================================================================

import os, sys, json, time, uuid, hashlib, logging, warnings
import subprocess, importlib.util, traceback, zipfile, io, random
from pathlib import Path
from datetime import datetime, timezone
from logging.handlers import RotatingFileHandler
from collections import defaultdict, Counter

warnings.filterwarnings("ignore")

# =============================================================================
# 1. DEPENDENCIES
# =============================================================================

DEPENDENCIES = {
    "numpy": "numpy",
    "pandas": "pandas",
    "sklearn": "scikit-learn",
    "joblib": "joblib",
    "plotly": "plotly"
}

for module, package in DEPENDENCIES.items():
    if importlib.util.find_spec(module) is None:
        print(f"Installing missing dependency: {package}")
        subprocess.check_call([
            sys.executable, "-m", "pip", "install", "-q", package
        ])

import numpy as np
import pandas as pd
import joblib
import plotly.express as px
import plotly.graph_objects as go

from IPython.display import display, HTML
from sklearn.ensemble import IsolationForest
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import (
    precision_score, recall_score, f1_score, confusion_matrix
)

# =============================================================================
# 2. CENTRAL CONFIGURATION
# =============================================================================

CONFIG = {
    "project_name": "GHOST SIGNAL AI",
    "version": "3.0.0",
    "random_seed": 42,
    "training_sessions": 800,
    "evaluation_sessions": 240,
    "contamination": 0.12,
    "high_priority_score": 60.0,
    "medium_priority_score": 35.0,
    "log_level": "INFO",
    "max_log_bytes": 2_000_000,
    "log_backups": 4,
    "output_directory": "ghost_signal_output",
    "strict_schema_validation": True,
    "timestamp_timezone": "UTC"
}

SEED = CONFIG["random_seed"]
np.random.seed(SEED)
random.seed(SEED)

RUN_ID = (
    datetime.now(timezone.utc).strftime("%Y%m%dT%H%M%SZ")
    + "_" + uuid.uuid4().hex[:8]
)

ROOT = Path(CONFIG["output_directory"])
LOG_DIR = ROOT / "logs"
DATA_DIR = ROOT / "data"
MODEL_DIR = ROOT / "models"
REPORT_DIR = ROOT / "reports"
CHART_DIR = ROOT / "charts"

for folder in [ROOT, LOG_DIR, DATA_DIR, MODEL_DIR, REPORT_DIR, CHART_DIR]:
    folder.mkdir(parents=True, exist_ok=True)

# =============================================================================
# 3. LOGGING INFRASTRUCTURE
# =============================================================================

class UTCFormatter(logging.Formatter):
    converter = time.gmtime

    def format(self, record):
        record.run_id = RUN_ID
        return super().format(record)

logger = logging.getLogger("ghost_signal")
logger.setLevel(getattr(logging, CONFIG["log_level"], logging.INFO))
logger.handlers.clear()
logger.propagate = False

formatter = UTCFormatter(
    "%(asctime)sZ | %(levelname)-8s | run=%(run_id)s | "
    "%(name)s | %(module)s:%(lineno)d | %(message)s",
    datefmt="%Y-%m-%dT%H:%M:%S"
)

console = logging.StreamHandler(sys.stdout)
console.setFormatter(formatter)
logger.addHandler(console)

file_handler = RotatingFileHandler(
    LOG_DIR / "ghost_signal.log",
    maxBytes=CONFIG["max_log_bytes"],
    backupCount=CONFIG["log_backups"],
    encoding="utf-8"
)
file_handler.setFormatter(formatter)
logger.addHandler(file_handler)

AUDIT_FILE = LOG_DIR / "audit.jsonl"

def audit(event, **details):
    record = {
        "timestamp_utc": datetime.now(timezone.utc).isoformat(),
        "run_id": RUN_ID,
        "event": event,
        **details
    }
    with open(AUDIT_FILE, "a", encoding="utf-8") as f:
        f.write(json.dumps(record, default=str) + "\n")

def log_error(message):
    logger.exception(message)
    audit(
        "exception",
        message=message,
        traceback=traceback.format_exc()
    )

def save_json(path, obj):
    path = Path(path)
    path.parent.mkdir(parents=True, exist_ok=True)
    with open(path, "w", encoding="utf-8") as f:
        json.dump(obj, f, indent=2, default=str)

class PipelineError(RuntimeError):
    pass

class Stage:
    """Context manager that records stage timing and failures."""
    def __init__(self, name):
        self.name = name

    def __enter__(self):
        self.started = time.perf_counter()
        logger.info("STAGE_START | %s", self.name)
        audit("stage_start", stage=self.name)
        return self

    def __exit__(self, exc_type, exc_value, tb):
        elapsed = round(time.perf_counter() - self.started, 4)
        if exc_type:
            logger.error(
                "STAGE_FAILED | %s | duration=%.4fs | error=%s",
                self.name, elapsed, exc_value
            )
            audit(
                "stage_failed", stage=self.name,
                duration_seconds=elapsed, error=str(exc_value)
            )
        else:
            logger.info(
                "STAGE_SUCCESS | %s | duration=%.4fs",
                self.name, elapsed
            )
            audit(
                "stage_success", stage=self.name,
                duration_seconds=elapsed
            )
        return False

logger.info("=" * 78)
logger.info("Starting %s version %s", CONFIG["project_name"], CONFIG["version"])
logger.info("Run ID: %s", RUN_ID)
audit("startup", config=CONFIG)

# =============================================================================
# 4. EVENT SCHEMA
# =============================================================================

CORE_WORKFLOW = [
    "request_received",
    "auth_check",
    "process_request",
    "database_query",
    "response_sent"
]

OPTIONAL_EVENTS = ["cache_hit", "retry_attempt"]

EVENT_META = {
    "request_received": ("GATEWAY", "ENTRY"),
    "auth_check": ("AUTH", "SECURITY"),
    "process_request": ("WORKER", "PROCESSING"),
    "database_query": ("DATABASE", "STORAGE"),
    "response_sent": ("API", "EXIT"),
    "cache_hit": ("WORKER", "OPTIMIZATION"),
    "retry_attempt": ("API", "RECOVERY")
}

REQUIRED_COLUMNS = [
    "session_id", "timestamp", "event", "service", "category",
    "latency_ms", "cpu_percent", "ground_truth",
    "injected_missing_event"
]

# =============================================================================
# 5. SYNTHETIC DATA GENERATOR
# =============================================================================

def generate_sessions(n_sessions, inject_anomalies=False, seed=42):
    """
    Generate fictional event sessions.

    Evaluation faults are deliberately injected to measure missing-event
    detection. They are not real incidents.
    """
    rng = np.random.default_rng(seed)
    rows = []
    base = pd.Timestamp("2026-01-01", tz="UTC")

    for i in range(n_sessions):
        session_id = f"SESSION_{i:05d}"
        timestamp = base + pd.Timedelta(minutes=i * 3)
        events = CORE_WORKFLOW.copy()

        if rng.random() < 0.18:
            events.insert(3, "cache_hit")

        if rng.random() < 0.06:
            events.insert(2, "retry_attempt")

        truth = "NORMAL"
        injected_missing = None

        if inject_anomalies and i % 8 == 0:
            candidates = [
                e for e in events
                if e in ["auth_check", "process_request", "database_query"]
            ]
            injected_missing = str(rng.choice(candidates))
            events.remove(injected_missing)
            truth = "MISSING_EVENT"

        delayed = inject_anomalies and i % 11 == 0
        if delayed:
            truth = (
                "MISSING_EVENT_AND_DELAY"
                if injected_missing else "DELAY"
            )

        for position, event in enumerate(events):
            service, category = EVENT_META[event]
            latency = float(rng.lognormal(3.5, 0.35))
            delay = float(rng.exponential(1.3))

            if delayed and position >= 2:
                latency += float(rng.uniform(180, 320))
                delay += float(rng.uniform(18, 35))

            timestamp += pd.Timedelta(seconds=delay)

            rows.append({
                "session_id": session_id,
                "timestamp": timestamp,
                "event": event,
                "service": service,
                "category": category,
                "latency_ms": latency,
                "cpu_percent": float(rng.uniform(15, 75)),
                "ground_truth": truth,
                "injected_missing_event": injected_missing
            })

    return pd.DataFrame(rows, columns=REQUIRED_COLUMNS)

# =============================================================================
# 6. DATA VALIDATION AND QUALITY AUDIT
# =============================================================================

def validate_data(df, dataset_name):
    if not isinstance(df, pd.DataFrame):
        raise PipelineError(f"{dataset_name} is not a DataFrame")

    missing = sorted(set(REQUIRED_COLUMNS) - set(df.columns))
    if missing:
        raise PipelineError(
            f"{dataset_name}: missing columns {missing}"
        )

    if df.empty:
        raise PipelineError(f"{dataset_name} is empty")

    df = df.copy()
    df["timestamp"] = pd.to_datetime(
        df["timestamp"], errors="coerce", utc=True
    )

    if df["timestamp"].isna().any():
        raise PipelineError(f"{dataset_name}: invalid timestamps")

    for col in ["session_id", "event", "service", "category"]:
        if df[col].isna().any():
            raise PipelineError(
                f"{dataset_name}: null values in {col}"
            )

    unknown_events = sorted(set(df["event"]) - set(EVENT_META))
    if unknown_events:
        raise PipelineError(f"Unknown events: {unknown_events}")

    for col in ["latency_ms", "cpu_percent"]:
        df[col] = pd.to_numeric(df[col], errors="coerce")
        if df[col].isna().any() or not np.isfinite(df[col]).all():
            raise PipelineError(f"Invalid numeric values in {col}")

    if (df["latency_ms"] < 0).any():
        raise PipelineError("Negative latency detected")

    if not df["cpu_percent"].between(0, 100).all():
        raise PipelineError("CPU values outside [0,100]")

    duplicate_count = int(
        df.duplicated(["session_id", "timestamp", "event"]).sum()
    )

    logger.info(
        "DATA_QUALITY | dataset=%s | rows=%d | sessions=%d "
        "| duplicates=%d | null_cells=%d",
        dataset_name,
        len(df),
        df["session_id"].nunique(),
        duplicate_count,
        int(df.isna().sum().sum())
    )

    if duplicate_count:
        logger.warning("%s contains duplicate event records", dataset_name)

    audit(
        "data_validation",
        dataset=dataset_name,
        rows=len(df),
        sessions=int(df["session_id"].nunique()),
        duplicate_count=duplicate_count
    )

    return df.sort_values(
        ["session_id", "timestamp"]
    ).reset_index(drop=True)

def hash_dataframe(df):
    return hashlib.sha256(
        pd.util.hash_pandas_object(df, index=True).values.tobytes()
    ).hexdigest()

# =============================================================================
# 7. TEMPORAL TRANSITION MODEL
# =============================================================================

class TransitionModel:
    def __init__(self, smoothing=0.01):
        self.smoothing = smoothing
        self.counts = defaultdict(Counter)
        self.probabilities = {}
        self.fitted = False

    def fit(self, df):
        self.counts = defaultdict(Counter)

        for _, group in df.groupby("session_id"):
            events = group.sort_values("timestamp")["event"].tolist()
            for a, b in zip(events[:-1], events[1:]):
                self.counts[a][b] += 1

        vocabulary = list(EVENT_META.keys())

        for source in vocabulary:
            target_counts = self.counts[source]
            denominator = (
                sum(target_counts.values())
                + self.smoothing * len(vocabulary)
            )

            self.probabilities[source] = {
                target: (
                    target_counts.get(target, 0) + self.smoothing
                ) / denominator
                for target in vocabulary
            }

        self.fitted = True
        logger.info(
            "TRANSITION_MODEL | edge_count=%d",
            sum(sum(c.values()) for c in self.counts.values())
        )
        audit(
            "transition_model_fitted",
            edge_count=sum(sum(c.values()) for c in self.counts.values())
        )
        return self

    def probability(self, source, target):
        return self.probabilities.get(source, {}).get(target, 0.001)

# =============================================================================
# 8. SESSION INVESTIGATION
# =============================================================================

def investigate_session(group, transition_model):
    group = group.sort_values("timestamp")
    events = group["event"].tolist()
    present = set(events)

    missing_core = [e for e in CORE_WORKFLOW if e not in present]
    skipped = []
    surprise_values = []

    for source, target in zip(events[:-1], events[1:]):
        p = transition_model.probability(source, target)
        surprise_values.append(-np.log(max(p, 1e-8)))

        if source in CORE_WORKFLOW and target in CORE_WORKFLOW:
            i_source = CORE_WORKFLOW.index(source)
            i_target = CORE_WORKFLOW.index(target)
            if i_target > i_source + 1:
                skipped.extend(
                    CORE_WORKFLOW[i_source + 1:i_target]
                )

    skipped = list(dict.fromkeys(skipped))
    duration = (
        (group["timestamp"].iloc[-1] - group["timestamp"].iloc[0])
        .total_seconds()
        if len(group) > 1 else 0.0
    )
    latency = float(group["latency_ms"].mean())
    surprise = float(np.mean(surprise_values)) if surprise_values else 0.0

    # Transparent heuristic, not a probability.
    score = min(
        100.0,
        38.0 * len(missing_core)
        + min(25.0, surprise * 8.0)
        + min(25.0, max(0.0, duration - 12.0) * 0.6)
        + min(12.0, max(0.0, latency - 50.0) * 0.05)
    )

    if missing_core:
        finding = "MISSING_EXPECTED_EVENT"
    elif duration > 30:
        finding = "UNUSUAL_SESSION_DURATION"
    elif surprise > 2:
        finding = "UNUSUAL_TRANSITION"
    else:
        finding = "NO_MAJOR_RULE_VIOLATION"

    explanations = []
    if missing_core:
        explanations.append(
            "Expected core events absent: " + ", ".join(missing_core)
        )
    if skipped:
        explanations.append(
            "Canonical workflow steps skipped: " + ", ".join(skipped)
        )
    if duration > 30:
        explanations.append(f"Observed duration {duration:.1f}s exceeds 30s")
    if latency > 100:
        explanations.append(f"Mean latency {latency:.1f}ms exceeds 100ms")
    if not explanations:
        explanations.append("No configured absence rule was triggered")

    truth = group["ground_truth"].dropna()
    truth = str(truth.iloc[0]) if len(truth) else "UNKNOWN"

    injected = group["injected_missing_event"].dropna()
    injected = str(injected.iloc[0]) if len(injected) else None

    return {
        "session_id": str(group["session_id"].iloc[0]),
        "events_observed": len(events),
        "missing_events": ", ".join(missing_core) if missing_core else "None",
        "skipped_events": ", ".join(skipped) if skipped else "None",
        "duration_seconds": round(duration, 3),
        "mean_latency_ms": round(latency, 3),
        "transition_surprise": round(surprise, 4),
        "absence_score": round(score, 2),
        "finding": finding,
        "explanation": " | ".join(explanations),
        "ground_truth": truth,
        "injected_missing_event": injected
    }

FEATURE_COLUMNS = [
    "events_observed", "duration_seconds",
    "mean_latency_ms", "transition_surprise", "absence_score"
]

# =============================================================================
# 9. EXPERIMENT TRACKING
# =============================================================================

EXPERIMENT = {
    "run_id": RUN_ID,
    "project": CONFIG["project_name"],
    "version": CONFIG["version"],
    "started_at_utc": datetime.now(timezone.utc).isoformat(),
    "config": CONFIG.copy(),
    "metrics": {},
    "status": "RUNNING"
}

def log_metric(name, value):
    if isinstance(value, (np.integer, np.floating)):
        value = value.item()
    EXPERIMENT["metrics"][name] = value
    logger.info("EXPERIMENT_METRIC | %s=%s", name, value)
    audit("experiment_metric", name=name, value=value)

# =============================================================================
# 10. AUTOMATED TESTING
# =============================================================================

def run_tests(training, evaluation, investigations, model):
    outcomes = []

    def check(name, condition, detail):
        outcomes.append({
            "test_name": name,
            "status": "PASS" if bool(condition) else "FAIL",
            "detail": detail
        })
        log = logger.info if condition else logger.error
        log("TEST_%s | %s | %s",
            "PASS" if condition else "FAIL", name, detail)

    check("training_nonempty", len(training) > 0, str(len(training)))
    check("evaluation_nonempty", len(evaluation) > 0, str(len(evaluation)))
    check(
        "one_investigation_per_session",
        len(investigations) == evaluation["session_id"].nunique(),
        f"investigations={len(investigations)}"
    )
    check(
        "scores_within_bounds",
        investigations["absence_score"].between(0, 100).all(),
        "Expected range 0..100"
    )
    check(
        "missing_events_detected",
        investigations["missing_events"].ne("None").any(),
        "At least one injected omission should be detected"
    )
    check(
        "model_fitted",
        hasattr(model, "estimators_"),
        "Isolation Forest fitted"
    )

    return pd.DataFrame(outcomes)

# =============================================================================
# 11. MAIN END-TO-END PIPELINE
# =============================================================================

def run_pipeline():
    with Stage("generate_and_validate_data"):
        training = validate_data(
            generate_sessions(CONFIG["training_sessions"], False, SEED),
            "training"
        )
        evaluation = validate_data(
            generate_sessions(
                CONFIG["evaluation_sessions"], True, SEED + 1
            ),
            "evaluation"
        )

        training_hash = hash_dataframe(training)
        evaluation_hash = hash_dataframe(evaluation)

        logger.info("TRAIN_HASH | %s", training_hash)
        logger.info("EVAL_HASH | %s", evaluation_hash)

    with Stage("learn_temporal_patterns"):
        transition_model = TransitionModel().fit(training)

    with Stage("investigate_sessions"):
        investigations = pd.DataFrame([
            investigate_session(group, transition_model)
            for _, group in evaluation.groupby("session_id")
        ])

        if investigations.empty:
            raise PipelineError("Investigation produced no results")

    with Stage("fit_unsupervised_ml_model"):
        X_raw = investigations[FEATURE_COLUMNS].astype(float).fillna(0.0)
        scaler = StandardScaler()
        X = scaler.fit_transform(X_raw)

        model = IsolationForest(
            n_estimators=200,
            contamination=CONFIG["contamination"],
            random_state=SEED,
            n_jobs=-1
        )
        model.fit(X)

        investigations["ml_anomaly"] = model.predict(X) == -1
        investigations["ml_anomaly_strength"] = -model.score_samples(X)

        investigations["flagged"] = (
            investigations["missing_events"].ne("None")
            | investigations["ml_anomaly"]
        )

        investigations["priority"] = np.select(
            [
                investigations["absence_score"]
                >= CONFIG["high_priority_score"],
                investigations["absence_score"]
                >= CONFIG["medium_priority_score"]
            ],
            ["HIGH", "MEDIUM"],
            default="LOW"
        )

    with Stage("evaluate_detection"):
        y_true = investigations["ground_truth"].str.contains(
            "MISSING_EVENT", regex=False
        ).astype(int)
        y_pred = investigations["missing_events"].ne("None").astype(int)

        metrics = {
            "precision": float(precision_score(y_true, y_pred, zero_division=0)),
            "recall": float(recall_score(y_true, y_pred, zero_division=0)),
            "f1": float(f1_score(y_true, y_pred, zero_division=0)),
            "confusion_matrix": confusion_matrix(
                y_true, y_pred, labels=[0, 1]
            ).tolist()
        }

        for name, value in metrics.items():
            log_metric(name, value)

    with Stage("automated_tests"):
        tests = run_tests(training, evaluation, investigations, model)

    with Stage("save_model_and_reports"):
        bundle = {
            "model": model,
            "scaler": scaler,
            "transition_model": transition_model,
            "feature_columns": FEATURE_COLUMNS,
            "config": CONFIG,
            "version": CONFIG["version"],
            "run_id": RUN_ID
        }

        model_path = MODEL_DIR / "ghost_signal_model.joblib"
        joblib.dump(bundle, model_path)

        training.to_csv(DATA_DIR / "training_events.csv", index=False)
        evaluation.to_csv(DATA_DIR / "evaluation_events.csv", index=False)
        investigations.to_csv(
            REPORT_DIR / "investigations.csv", index=False
        )
        tests.to_csv(REPORT_DIR / "automated_tests.csv", index=False)

        summary = {
            "project": CONFIG["project_name"],
            "version": CONFIG["version"],
            "run_id": RUN_ID,
            "data_type": "synthetic",
            "training_sessions": int(training["session_id"].nunique()),
            "evaluation_sessions": int(evaluation["session_id"].nunique()),
            "flagged_sessions": int(investigations["flagged"].sum()),
            "missing_event_sessions": int(
                investigations["missing_events"].ne("None").sum()
            ),
            "mean_absence_score": float(
                investigations["absence_score"].mean()
            ),
            "metrics": metrics,
            "training_sha256": training_hash,
            "evaluation_sha256": evaluation_hash,
            "model_path": str(model_path),
            "limitations": [
                "Synthetic demonstration data only",
                "Absence score is a heuristic, not a probability",
                "Anomaly detection does not prove causation",
                "Optional workflow steps require system-specific configuration",
                "Isolation Forest is fit on evaluation features in this demo; "
                "production evaluation should use a separate calibration set"
            ]
        }

        EXPERIMENT["status"] = "SUCCESS"
        EXPERIMENT["finished_at_utc"] = datetime.now(timezone.utc).isoformat()
        EXPERIMENT["metrics"].update(metrics)
        EXPERIMENT["data_hashes"] = {
            "training": training_hash,
            "evaluation": evaluation_hash
        }

        save_json(REPORT_DIR / "summary.json", summary)
        save_json(REPORT_DIR / "experiment.json", EXPERIMENT)

    return {
        "training": training,
        "evaluation": evaluation,
        "investigations": investigations,
        "tests": tests,
        "summary": summary,
        "bundle": bundle,
        "metrics": metrics
    }

# =============================================================================
# 12. EXECUTION
# =============================================================================

try:
    results = run_pipeline()
    investigations = results["investigations"]
    summary = results["summary"]
    tests = results["tests"]

    logger.info("PIPELINE_COMPLETED | status=SUCCESS")

    # -------------------------------------------------------------------------
    # VISUAL DASHBOARD
    # -------------------------------------------------------------------------

    display(HTML("""
    <div style="background:linear-gradient(125deg,#07111f,#172b46);
      color:white;padding:26px;border-radius:18px;font-family:Arial">
      <div style="color:#5eead4;font-size:12px;letter-spacing:4px">
        TEMPORAL INTELLIGENCE LAB
      </div>
      <h1 style="margin:10px 0;font-size:32px">👻 GHOST SIGNAL AI</h1>
      <p style="color:#cbd5e1;margin-bottom:0">
        Event absence detection · Temporal investigation · Explainable ML
      </p>
    </div>
    """))

    cards = [
        ("Sessions analyzed", summary["evaluation_sessions"]),
        ("Flagged sessions", summary["flagged_sessions"]),
        ("Missing-event sessions", summary["missing_event_sessions"]),
        ("Mean score", round(summary["mean_absence_score"], 2))
    ]

    html_cards = "".join(
        f"""
        <div style="flex:1;min-width:150px;padding:18px;border-radius:13px;
          background:#111c2e;border:1px solid #293952;color:white">
          <div style="color:#94a3b8;font-size:12px">{label}</div>
          <div style="font-size:27px;font-weight:bold;margin-top:8px">{value}</div>
        </div>
        """
        for label, value in cards
    )

    display(HTML(
        '<div style="display:flex;flex-wrap:wrap;gap:12px;margin:16px 0">'
        + html_cards + "</div>"
    ))

    fig = px.histogram(
        investigations,
        x="absence_score",
        color="priority",
        nbins=20,
        title="Distribution of Heuristic Absence Scores",
        labels={"absence_score": "Score (0–100)"},
        color_discrete_sequence=["#14b8a6", "#f59e0b", "#ef4444"]
    )
    fig.update_layout(template="plotly_dark", height=430)
    fig.show()

    # Learned event relationship graph.
    graph = nx.DiGraph()
    for source, targets in results["bundle"]["transition_model"].counts.items():
        for target, count in targets.items():
            graph.add_edge(source, target, weight=count)

    pos = nx.spring_layout(graph, seed=SEED)
    edge_x, edge_y = [], []

    for source, target in graph.edges():
        x0, y0 = pos[source]
        x1, y1 = pos[target]
        edge_x.extend([x0, x1, None])
        edge_y.extend([y0, y1, None])

    node_x = [pos[n][0] for n in graph.nodes()]
    node_y = [pos[n][1] for n in graph.nodes()]

    edge_trace = go.Scatter(
        x=edge_x, y=edge_y, mode="lines",
        line=dict(width=1, color="#64748b"),
        hoverinfo="none"
    )

    node_trace = go.Scatter(
        x=node_x, y=node_y, mode="markers+text",
        text=list(graph.nodes()),
        textposition="bottom center",
        marker=dict(size=24, color="#14b8a6",
                    line=dict(width=1, color="white")),
        hoverinfo="text"
    )

    fig_graph = go.Figure([edge_trace, node_trace])
    fig_graph.update_layout(
        title="Learned Event Transition Graph",
        template="plotly_dark",
        height=500,
        showlegend=False,
        xaxis=dict(visible=False),
        yaxis=dict(visible=False)
    )
    fig_graph.show()

    print("\nTOP INVESTIGATION RESULTS")
    display(
        investigations.sort_values(
            ["absence_score", "ml_anomaly_strength"],
            ascending=False
        )[
            [
                "session_id", "priority", "absence_score", "finding",
                "missing_events", "skipped_events",
                "duration_seconds", "ml_anomaly", "explanation"
            ]
        ].head(20).reset_index(drop=True)
    )

    print("\nAUTOMATED TESTS")
    display(tests)

    print("\nSYNTHETIC EVALUATION METRICS")
    display(pd.DataFrame([
        {"metric": k, "value": v}
        for k, v in results["metrics"].items()
        if k != "confusion_matrix"
    ]))

    # -------------------------------------------------------------------------
    # EXPORT COMPLETE ARTIFACT BUNDLE
    # -------------------------------------------------------------------------

    zip_path = ROOT / "ghost_signal_complete_artifacts.zip"

    with zipfile.ZipFile(zip_path, "w", zipfile.ZIP_DEFLATED) as archive:
        for file_path in ROOT.rglob("*"):
            if file_path.is_file() and file_path != zip_path:
                archive.write(file_path, file_path.relative_to(ROOT))

    print("\n" + "=" * 78)
    print("GHOST SIGNAL AI — COMPLETE")
    print("=" * 78)
    print(f"Run ID: {RUN_ID}")
    print(f"Model: {MODEL_DIR / 'ghost_signal_model.joblib'}")
    print(f"Logs: {LOG_DIR / 'ghost_signal.log'}")
    print(f"Audit: {AUDIT_FILE}")
    print(f"Reports: {REPORT_DIR}")
    print(f"Complete artifact ZIP: {zip_path}")
    print("\nDownload the ZIP from the Colab Files panel or the Jupyter file browser.")
    print("Remember: all evaluation results apply to synthetic data only.")

except Exception as exc:
    logger.exception("FATAL_PIPELINE_ERROR")
    EXPERIMENT["status"] = "FAILED"
    EXPERIMENT["error"] = str(exc)
    EXPERIMENT["traceback"] = traceback.format_exc()

    save_json(REPORT_DIR / "failed_experiment.json", EXPERIMENT)

    print("\nGHOST SIGNAL AI FAILED")
    print("Check the log and failed_experiment.json for the full traceback.")
    raise


2026-10-13T06:16:18Z | INFO     | run=20261013T061618Z_3063a90b | ghost_signal | 2146306289:203 | ==============================================================================
2026-10-13T06:16:18Z | INFO     | run=20261013T061618Z_3063a90b | ghost_signal | 2146306289:204 | Starting GHOST SIGNAL AI version 3.0.0
2026-10-13T06:16:18Z | INFO     | run=20261013T061618Z_3063a90b | ghost_signal | 2146306289:205 | Run ID: 20261013T061618Z_3063a90b
2026-10-13T06:16:18Z | INFO     | run=20261013T061618Z_3063a90b | ghost_signal | 2146306289:177 | STAGE_START | generate_and_validate_data
2026-10-13T06:16:18Z | INFO     | run=20261013T061618Z_3063a90b | ghost_signal | 2146306289:358 | DATA_QUALITY | dataset=training | rows=4185 | sessions=800 | duplicates=0 | null_cells=4185
2026-10-13T06:16:18Z | INFO     | run=20261013T061618Z_3063a90b | ghost_signal | 2146306289:358 | DATA_QUALITY | dataset=evaluation | rows=1238 | sessions=240 | duplicates=0 | null_cells=1109
2026-10-13T06:16:18Z | INFO     |

2026-10-13T06:16:27Z | ERROR    | run=20261013T061618Z_3063a90b | ghost_signal | 2146306289:908 | FATAL_PIPELINE_ERROR
Traceback (most recent call last):
  File "/tmp/ipykernel_1712/2146306289.py", line 817, in <cell line: 0>
    graph = nx.DiGraph()
            ^^
NameError: name 'nx' is not defined. Did you mean: 'np'?

GHOST SIGNAL AI FAILED
Check the log and failed_experiment.json for the full traceback.


NameError: name 'nx' is not defined